In [2]:
import pandas as pd
import re
import os

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

RAW_DIR = "../data/raw"
OUT_DIR = "../data/processed"
os.makedirs(OUT_DIR, exist_ok=True)

def snake(col):
    col = col.strip()
    col = re.sub(r"[^\w\s]", "", col)
    col = re.sub(r"\s+", "_", col)
    return col.lower()

def to_num(series):
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce"
    )

#Quick look at one file before cleaning everything

In [3]:
df = pd.read_csv(f"{RAW_DIR}/data_centers.csv")
df.head()

,Name,Current H100 equivalents,Current power (MW),Current total capital cost (2025 USD billions),Owner,Users,Selected Sources,Calculations sheet,Project,Current chip types,All chip types,Investors,Construction companies,Energy companies,Country,Address
0,Colossus 2,1.111673e+06,946.0,35.836372,SpaceXAI #confident,"Anthropic #confident, Cursor #confident, Space...",- [WSJ profile of the Colossus data centers](h...,https://docs.google.com/spreadsheets/d/1_FQATV...,NaN,"B200,B300","B200,B300",NaN,NaN,NaN,United States,"5420 Tulane Rd, Memphis, TN 38109"
1,Microsoft Fairwater Atlanta,7.687687e+05,636.0,24.092952,Microsoft #confident,"OpenAI #likely, Microsoft #likely",- [2023 Air Permit application](https://drive....,https://docs.google.com/spreadsheets/d/1XeruHu...,Microsoft AI WAN #confident,B200,B200,NaN,NaN,Georgia Power,United States,"1435 Hwy 54 W, Fayetteville, GA 30214"
2,Meta Prometheus,7.630116e+05,631.0,23.903542,Meta #confident,Meta #confident,- [Meta blog post with details on Prometheus](...,https://docs.google.com/spreadsheets/d/1rL6xWR...,Prometheus #confident,B200,B200,NaN,NaN,"Williams,American Electric Power",United States,"1 Community Cir, New Albany, OH 43054"
3,Anthropic-Amazon New Carlisle,6.859121e+05,910.0,34.472620,Amazon #confident,Anthropic #confident,- [Air Construction Permit application for the...,https://docs.google.com/spreadsheets/d/1umU-KL...,Project Rainier #confident,Trainium2,Trainium2,NaN,NaN,NaN,United States,"55001 Larrison Blvd, New Carlisle, IN 46552"
4,OpenAI Stargate Abilene,5.093482e+05,421.0,15.948322,Oracle #confident,OpenAI #confident,"- [Crusoe 2024 Impact Report, with details on ...",https://docs.google.com/spreadsheets/d/197BW-5...,Stargate #confident,"B200,B300","B200,B300","Softbank,OpenAI,Oracle",Mortenson,American Electric Power,United States,"5502 Spinks Rd, Abilene, TX 79601"


#Cleaning rules for all 6 files

In [4]:
FILES = {
    "data_centers.csv": {
        "numeric": ["Current H100 equivalents", "Current power (MW)",
                    "Current total capital cost (2025 USD billions)"],
        "dates": [],
    },
    "data_center_timelines.csv": {
        "numeric": ["Buildings operational", "IT power (MW)", "Power (MW)",
                    "H100 equivalents", "Performance (8-bit OP/s)",
                    "Total capital cost (2025 USD billions)",
                    "Compute cost (2025 USD billions)",
                    "Construction cost (2025 USD billions)",
                    "Annual operating cost (2025 USD billions)",
                    "Water use (MGD)",
                    "Current H100 equivalents (from Data center)"],
        "dates": ["Date"],
    },
    "data_center_chip_quantities.csv": {
        "numeric": ["Number of Units"],
        "dates": ["Date", "Created", "Last Modified"],
    },
    "data_center_chillers.csv": {
        "numeric": ["Cooling capacity (kW)", "Number of fans",
                    "Length (m)", "Width (m)", "Area (m^2)"],
        "dates": [],
    },
    "data_center_cooling_towers.csv": {
        "numeric": ["Length (m)", "Width (m)", "Height (m)", "Area (m^2)",
                    "Cooling capacity (stated, kW)",
                    "Cooling capacity (central estimate, kW)",
                    "Fan Diameter (m)", "Number of Fans",
                    "Fan estimate diameter (m)"],
        "dates": [],
    },
    "gpu_clusters.csv": {
        "numeric": ["Max OP/s (log)", "H100 equivalents",
                    "Chip quantity (primary)", "Power Capacity (MW)",
                    "Total number of AI chips", "latitude", "longitude"],
        "dates": ["First Operational Date"],
    },
}

#Clean and save all 6 at once

In [5]:
for filename, rules in FILES.items():
    df = pd.read_csv(f"{RAW_DIR}/{filename}")

    for col in rules["numeric"]:
        if col in df.columns:
            df[col] = to_num(df[col])

    for col in rules["dates"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    df.columns = [snake(c) for c in df.columns]

    out_name = filename.replace(".csv", "_clean.csv")
    df.to_csv(f"{OUT_DIR}/{out_name}", index=False)
    print(f"{filename} -> {out_name}  {df.shape}")

data_centers.csv -> data_centers_clean.csv  (74, 16)
data_center_timelines.csv -> data_center_timelines_clean.csv  (419, 14)
data_center_chip_quantities.csv -> data_center_chip_quantities_clean.csv  (196, 14)
data_center_chillers.csv -> data_center_chillers_clean.csv  (143, 12)
data_center_cooling_towers.csv -> data_center_cooling_towers_clean.csv  (527, 17)
gpu_clusters.csv -> gpu_clusters_clean.csv  (482, 57)
